# Building charGPT from scratch

This notebook builds a character-level GPT step by step, following the same path as
Andrej Karpathy's *"Let's build GPT"*:

1. **Data** — a character-level tokenizer on Tiny Shakespeare
2. **Bigram baseline** — the simplest possible language model
3. **Self-attention** — the key idea: tokens talk to their past
4. **Multi-head + feed-forward + blocks** — the full Transformer
5. **Train and sample** — put it all together with the packaged `GPT`

The goal is intuition: by the end, every line in [`chargpt/model.py`](../chargpt/model.py)
should feel obvious.

## 0. Setup

Make sure the dataset is present (`python data/download.py`) and the package is
installed (`pip install -e ".[dev]"`).

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
device

## 1. Data: a character-level tokenizer

The vocabulary is just the set of unique characters. Each character maps to an
integer id, and back. No subword magic.

In [ ]:
with open('../data/input.txt', 'r') as f:
    text = f.read()

print(f'length: {len(text):,} characters')
print(text[:200])

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

print('vocab_size:', vocab_size)
print('chars:', ''.join(chars))
print(encode('hello'), '->', decode(encode('hello')))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 8   # small for now, so we can print things
batch_size = 4

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs :', xb.shape)
print('targets:', yb.shape)

Notice `y` is `x` shifted by one: at every position the model is asked to predict
the **next** character. One (context, target) block is really `block_size`
training examples at once.

## 2. Bigram baseline

The simplest language model: a lookup table where token *i* directly stores the
logits for the next token. No context beyond the single previous character.

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_emb(idx)  # (B, T, vocab_size)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

m = BigramLM(vocab_size)
_, loss = m(xb, yb)
print('initial loss:', loss.item(), '| expected ~', round(torch.log(torch.tensor(vocab_size)).item(), 2))

The loss starts near `ln(vocab_size)` — the entropy of a uniform guess. A trained
bigram gets a bit better, but it can only ever look at one character of context.
To do better we need tokens to **see their history**.

## 3. The core trick: self-attention

Start with the crudest way for a token to use its past — average all previous
tokens together. This "bag of history" is a weighted sum where the weights happen
to be uniform.

In [ ]:
# A lower-triangular matrix of averaging weights turns 'mean of the past'
# into a single matrix multiply.
T = 4
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
print(wei)

Self-attention replaces those **uniform** weights with **learned, data-dependent**
ones. Every token emits a `query` (what am I looking for?) and a `key` (what do I
offer?). The affinity between token *t*'s query and token *s*'s key decides how
much *t* attends to *s*. We mask out the future (`tril`) and softmax into weights,
then take a weighted sum of `value`s.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k, q, v = key(x), query(x), value(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5    # (B, T, T) scaled affinities

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))     # causal mask: no peeking ahead
wei = F.softmax(wei, dim=-1)
out = wei @ v                                        # (B, T, head_size)

print('attention weights for one example (row t attends to columns <= t):')
print(wei[0].round(decimals=2))
print('output shape:', out.shape)

That block is exactly `Head` in `chargpt/model.py`. Two details worth naming:

- **The `sqrt(head_size)` scaling** keeps the pre-softmax scores from getting so
  large that softmax collapses onto one token.
- **The mask lives in a `register_buffer`**, not a plain attribute — it isn't a
  trained parameter, but it must travel with the model across `.to(device)` and
  `state_dict`.

## 4. The full Transformer

Stack the pieces:

- **Multi-head attention** — run several heads in parallel and concatenate, so
  the model attends to different things at once.
- **Feed-forward** — a per-token MLP that lets each position "think" after gathering
  context.
- **Block** — attention + feed-forward, each wrapped in a residual connection with
  pre-LayerNorm, so we can stack many of them and still train.

All of this is already packaged, so let's just import it.

In [ ]:
from chargpt import GPT, GPTConfig

config = GPTConfig(vocab_size=vocab_size, block_size=128, n_embd=192,
                   n_head=6, n_layer=6, dropout=0.1)
model = GPT(config).to(device)
print(f'{model.num_params() / 1e6:.2f}M parameters')

## 5. Train and sample

A standard AdamW loop. On CPU this is slow; drop `max_iters` if you just want to
see the mechanics. For a real run, use the CLI: `python -m chargpt.train`.

In [ ]:
from chargpt import get_batch as get_batch_pkg

train_data = train_data.to(device)
val_data = val_data.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

max_iters = 500    # bump to ~5000 for readable output
for step in range(max_iters):
    xb, yb = get_batch_pkg(train_data, config.block_size, batch_size=32, device=device)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f'step {step:4d} | loss {loss.item():.4f}')

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(idx, max_new_tokens=500)[0].tolist()))

After only 500 steps it is still mostly noise. Train for a few thousand steps
(the CLI defaults) and the samples start to look like Shakespeare — line breaks,
character names, and plausible-looking words emerge from a model that has only
ever seen single characters.

```bash
python -m chargpt.train --steps 5000
python -m chargpt.sample --tokens 500 --prompt "ROMEO:"
```